# APEMAP data overview

This notebook describes the canonical DuckDB database and the fixed parliament snapshot dates used by the analysis package.

In [1]:
# ruff: noqa: E402
import os
import sys
from pathlib import Path

root = Path(os.environ.get("APEMAP_PROJECT_ROOT", Path.cwd())).resolve()
while not (root / "pyproject.toml").exists() and root != root.parent:
    root = root.parent
sys.path.insert(0, str(root))

from apemap.constants import PARLIAMENT_METADATA
from apemap.db import get_connection

db_path = Path(
    os.environ.get("APEMAP_DB_PATH", root / "data" / "aped.duckdb")
).resolve()
conn = get_connection(db_path, read_only=True)

In [3]:
table_names = [row[0] for row in conn.execute("SHOW TABLES").fetchall()]
table_counts = {}
for name in table_names:
    row = conn.execute(f"SELECT COUNT(*) FROM {name}").fetchone()
    table_counts[name] = int(row[0]) if row is not None else 0
table_counts

{'institutions': 11247,
 'member_aph_46': 238,
 'member_aph_47': 237,
 'member_aph_48': 230,
 'member_education': 469,
 'member_secondary_school_education_46': 350,
 'member_secondary_school_education_47': 353,
 'member_secondary_school_education_48': 324,
 'members': 335,
 'parliament_service': 705,
 'school_finances_2021': 260,
 'school_snapshots': 9709,
 'v_coverage_metrics': 3,
 'v_member_secondary_education': 1027,
 'v_parliament_members': 705,
 'v_parliament_members_current': 662,
 'v_parliament_members_opening': 683}

In [4]:
snapshot_dates = {
    parliament: metadata["opening_date"]
    for parliament, metadata in PARLIAMENT_METADATA.items()
}
snapshot_dates

{46: '2019-07-02', 47: '2022-07-26', 48: '2025-07-22'}

In [5]:
conn.close()